# 实验 B：七类情绪模型的类别差异与 Neutral 专项评估

固定模型：

- `j-hartmann/emotion-english-distilroberta-base`

固定测试数据：

- MELD 官方 `test_sent_emo.csv`
- 七类标签：`anger / disgust / fear / joy / neutral / sadness / surprise`

本 Notebook 重点回答：

1. 同一个七分类模型对不同情绪的 Recall、Precision 和 F1 差异有多大？
2. `neutral` 的识别能力如何？
3. 模型是否容易把有情绪文本误判为中性？
4. 模型是否容易把中性文本“过度解读”为某种情绪？
5. 哪些类别之间最容易混淆？

> 重要限制：该模型的训练数据包含 MELD。因此本实验属于**同源数据评估**，适合观察类别差异和错误结构，不应被解释为完全独立的跨领域泛化结果。

In [ ]:
!pip -q install -U transformers accelerate scikit-learn scipy pandas==2.2.2 matplotlib

In [ ]:
import random
import warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from IPython.display import display
from scipy.stats import chi2_contingency
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from transformers import AutoModelForSequenceClassification, AutoTokenizer

SEED = 42
MODEL_NAME = "j-hartmann/emotion-english-distilroberta-base"

# MELD 官方仓库中的测试标注文件。
TEST_CSV_URL = (
    "https://raw.githubusercontent.com/declare-lab/MELD/"
    "master/data/MELD/test_sent_emo.csv"
)

BATCH_SIZE = 64
MAX_LENGTH = 128

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("运行设备：", device)

## 1. 读取 MELD 官方测试集

只使用每条语句本身，不使用音频、视频或上下文。  
MELD 的 `Emotion` 列提供七类真实标签。

In [ ]:
test_df = pd.read_csv(TEST_CSV_URL)

required_columns = {"Utterance", "Emotion"}
missing_columns = required_columns - set(test_df.columns)
if missing_columns:
    raise ValueError(f"数据缺少必要字段：{sorted(missing_columns)}")

test_df = test_df.dropna(subset=["Utterance", "Emotion"]).copy()
test_df["text"] = test_df["Utterance"].astype(str).str.strip()
test_df["true_label"] = test_df["Emotion"].astype(str).str.strip().str.lower()
test_df = test_df[test_df["text"].str.len() > 0].reset_index(drop=True)

expected_labels = {
    "anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"
}
observed_labels = set(test_df["true_label"].unique())

if observed_labels != expected_labels:
    raise ValueError(
        "MELD 标签与预期不一致。\n"
        f"实际：{sorted(observed_labels)}\n"
        f"预期：{sorted(expected_labels)}"
    )

print("测试样本数：", len(test_df))
display(
    test_df["true_label"]
    .value_counts()
    .rename_axis("emotion")
    .reset_index(name="support")
)
display(test_df[["text", "true_label"]].head())

## 2. 加载模型并校验标签

不假设模型输出编号与数据集编号相同，而是通过标签名称进行对齐。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()

def normalize_label(value):
    return str(value).strip().lower().replace(" ", "_")

model_id2label = {
    int(k): normalize_label(v)
    for k, v in model.config.id2label.items()
}

if set(model_id2label.values()) != expected_labels:
    raise ValueError(
        "模型标签与 MELD 七类标签不一致。\n"
        f"模型标签：{sorted(model_id2label.values())}"
    )

label_names = [
    "anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"
]
label_to_id = {label: i for i, label in enumerate(label_names)}

print("模型标签映射：", model_id2label)

## 3. 批量推理

In [ ]:
texts = test_df["text"].tolist()
pred_model_ids = []
confidences = []
all_probabilities = []

for start in range(0, len(texts), BATCH_SIZE):
    batch_texts = texts[start : start + BATCH_SIZE]
    encoded = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(device)

    with torch.inference_mode():
        logits = model(**encoded).logits
        probs = torch.softmax(logits, dim=-1)
        batch_conf, batch_pred = probs.max(dim=-1)

    pred_model_ids.extend(batch_pred.cpu().numpy().tolist())
    confidences.extend(batch_conf.cpu().numpy().tolist())
    all_probabilities.append(probs.cpu().numpy())

all_probabilities = np.vstack(all_probabilities)
pred_label_names = [model_id2label[int(i)] for i in pred_model_ids]

test_df["pred_label"] = pred_label_names
test_df["confidence"] = np.asarray(confidences)
test_df["correct"] = test_df["true_label"] == test_df["pred_label"]

y_true = test_df["true_label"].map(label_to_id).to_numpy()
y_pred = test_df["pred_label"].map(label_to_id).to_numpy()

display(test_df[["text", "true_label", "pred_label", "confidence", "correct"]].head())

## 4. 总体性能

In [ ]:
overall_df = pd.DataFrame([{
    "accuracy": accuracy_score(y_true, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    "macro_f1": f1_score(y_true, y_pred, average="macro"),
    "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    "emotion_only_accuracy": accuracy_score(
        y_true[test_df["true_label"].ne("neutral")],
        y_pred[test_df["true_label"].ne("neutral")],
    ),
    "n_test": len(test_df),
}])

display(overall_df.style.format({
    "accuracy": "{:.2%}",
    "balanced_accuracy": "{:.2%}",
    "macro_f1": "{:.2%}",
    "weighted_f1": "{:.2%}",
    "emotion_only_accuracy": "{:.2%}",
}))

## 5. 每类“分类准确率”

这里把每一种真实情绪的“分类准确率”定义为该类 Recall：

\[
\mathrm{ClassAccuracy}_i =
\frac{\mathrm{TP}_i}{\mathrm{TP}_i+\mathrm{FN}_i}
\]

它表示该类真实语料中有多少被正确识别。  
相比 one-vs-rest accuracy，它不会因为大量真负例而虚高。

In [ ]:
def wilson_interval(correct, total, z=1.96):
    if total == 0:
        return np.nan, np.nan
    p = correct / total
    denominator = 1 + z**2 / total
    center = (p + z**2 / (2 * total)) / denominator
    margin = (
        z
        * np.sqrt(p * (1 - p) / total + z**2 / (4 * total**2))
        / denominator
    )
    return max(0.0, center - margin), min(1.0, center + margin)

label_ids = list(range(len(label_names)))
cm = confusion_matrix(y_true, y_pred, labels=label_ids)

report = classification_report(
    y_true,
    y_pred,
    labels=label_ids,
    target_names=label_names,
    output_dict=True,
    zero_division=0,
)

rows = []
n_total = len(test_df)

for i, label in enumerate(label_names):
    tp = int(cm[i, i])
    support = int(cm[i, :].sum())
    predicted_count = int(cm[:, i].sum())
    fn = support - tp
    fp = predicted_count - tp
    tn = n_total - tp - fn - fp
    ci_low, ci_high = wilson_interval(tp, support)

    class_data = test_df[test_df["true_label"] == label]
    wrong_data = class_data[~class_data["correct"]]

    rows.append({
        "emotion": label,
        "support": support,
        "correct": tp,
        "incorrect": fn,
        "class_accuracy_recall": tp / support if support else np.nan,
        "ci95_low": ci_low,
        "ci95_high": ci_high,
        "precision": report[label]["precision"],
        "f1": report[label]["f1-score"],
        "one_vs_rest_accuracy": (tp + tn) / n_total,
        "mean_confidence": class_data["confidence"].mean(),
        "mean_wrong_confidence": wrong_data["confidence"].mean(),
    })

per_class_df = (
    pd.DataFrame(rows)
    .sort_values("class_accuracy_recall", ascending=False)
    .reset_index(drop=True)
)

display(per_class_df.style.format({
    "class_accuracy_recall": "{:.2%}",
    "ci95_low": "{:.2%}",
    "ci95_high": "{:.2%}",
    "precision": "{:.2%}",
    "f1": "{:.2%}",
    "one_vs_rest_accuracy": "{:.2%}",
    "mean_confidence": "{:.3f}",
    "mean_wrong_confidence": "{:.3f}",
}))

## 6. Neutral 专项指标

定义：

\[
\mathrm{FalseNeutralRate} =
\frac{\text{真实非中性但预测为 neutral}}
{\text{真实非中性样本数}}
\]

\[
\mathrm{FalseEmotionalRate} =
\frac{\text{真实 neutral 但预测为非中性}}
{\text{真实 neutral 样本数}}
\]

- False Neutral Rate 高：模型容易漏掉真实情绪。
- False Emotional Rate 高：模型容易过度解读客观或平淡文本。

In [ ]:
true_neutral = test_df["true_label"].eq("neutral")
pred_neutral = test_df["pred_label"].eq("neutral")

neutral_tp = int((true_neutral & pred_neutral).sum())
neutral_fp = int((~true_neutral & pred_neutral).sum())
neutral_fn = int((true_neutral & ~pred_neutral).sum())
neutral_tn = int((~true_neutral & ~pred_neutral).sum())

neutral_precision = neutral_tp / (neutral_tp + neutral_fp) if neutral_tp + neutral_fp else np.nan
neutral_recall = neutral_tp / (neutral_tp + neutral_fn) if neutral_tp + neutral_fn else np.nan
neutral_f1 = (
    2 * neutral_precision * neutral_recall / (neutral_precision + neutral_recall)
    if neutral_precision + neutral_recall
    else 0.0
)

false_neutral_rate = neutral_fp / (~true_neutral).sum()
false_emotional_rate = neutral_fn / true_neutral.sum()
predicted_to_true_neutral_ratio = pred_neutral.sum() / true_neutral.sum()

neutral_metrics_df = pd.DataFrame([{
    "neutral_support": int(true_neutral.sum()),
    "predicted_neutral": int(pred_neutral.sum()),
    "neutral_precision": neutral_precision,
    "neutral_recall": neutral_recall,
    "neutral_f1": neutral_f1,
    "false_neutral_rate": false_neutral_rate,
    "false_emotional_rate": false_emotional_rate,
    "predicted_to_true_neutral_ratio": predicted_to_true_neutral_ratio,
}])

display(neutral_metrics_df.style.format({
    "neutral_precision": "{:.2%}",
    "neutral_recall": "{:.2%}",
    "neutral_f1": "{:.2%}",
    "false_neutral_rate": "{:.2%}",
    "false_emotional_rate": "{:.2%}",
    "predicted_to_true_neutral_ratio": "{:.3f}",
}))

## 7. Neutral 被误判成了什么？哪些情绪被误判成 Neutral？

In [ ]:
neutral_to_emotion = (
    test_df[true_neutral & ~pred_neutral]
    .groupby("pred_label")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

emotion_to_neutral = (
    test_df[~true_neutral & pred_neutral]
    .groupby("true_label")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("真实 neutral 被模型判成：")
display(neutral_to_emotion)

print("真实情绪被模型判成 neutral：")
display(emotion_to_neutral)

## 8. 类别差异统计检验

使用“真实类别 × 是否预测正确”的卡方检验：

- `p < 0.05`：至少部分情绪类别的正确率存在统计显著差异。
- Cramér's V：类别与是否预测正确之间关联的效应量。

In [ ]:
correct_incorrect_table = np.column_stack([
    np.diag(cm),
    cm.sum(axis=1) - np.diag(cm),
])

chi2, p_value, dof, expected = chi2_contingency(correct_incorrect_table)
n = correct_incorrect_table.sum()
cramers_v = np.sqrt(chi2 / n)

significance_df = pd.DataFrame([{
    "chi_square": chi2,
    "degrees_of_freedom": dof,
    "p_value": p_value,
    "cramers_v": cramers_v,
    "significant_at_0.05": p_value < 0.05,
}])

display(significance_df.style.format({
    "chi_square": "{:.4f}",
    "p_value": "{:.6g}",
    "cramers_v": "{:.4f}",
}))

## 9. 每类 Recall 与 95% Wilson 置信区间

In [ ]:
plot_df = per_class_df.sort_values("class_accuracy_recall", ascending=False).copy()

x = np.arange(len(plot_df))
values = plot_df["class_accuracy_recall"].to_numpy()
lower_errors = values - plot_df["ci95_low"].to_numpy()
upper_errors = plot_df["ci95_high"].to_numpy() - values

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x, values)
ax.errorbar(
    x,
    values,
    yerr=np.vstack([lower_errors, upper_errors]),
    fmt="none",
    capsize=4,
)
ax.set_xticks(x)
ax.set_xticklabels(plot_df["emotion"], rotation=30, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Recall / class accuracy")
ax.set_title("MELD test: per-emotion recall with 95% Wilson CI")
ax.grid(axis="y", alpha=0.3)

for i, value in enumerate(values):
    ax.text(i, min(value + upper_errors[i] + 0.025, 1.03), f"{value:.1%}",
            ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("experiment_b_per_class_recall.png", dpi=200, bbox_inches="tight")
plt.show()

## 10. 归一化混淆矩阵

In [ ]:
row_sums = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(
    cm,
    row_sums,
    out=np.zeros_like(cm, dtype=float),
    where=row_sums != 0,
)

fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(cm_norm, aspect="auto")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

ax.set_xticks(np.arange(len(label_names)))
ax.set_yticks(np.arange(len(label_names)))
ax.set_xticklabels(label_names, rotation=45, ha="right")
ax.set_yticklabels(label_names)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("MELD test: row-normalized confusion matrix")

threshold = cm_norm.max() / 2
for i in range(cm_norm.shape[0]):
    for j in range(cm_norm.shape[1]):
        value = cm_norm[i, j]
        ax.text(
            j, i, f"{value:.1%}\n(n={cm[i, j]})",
            ha="center", va="center",
            color="white" if value > threshold else "black",
            fontsize=8,
        )

plt.tight_layout()
plt.savefig("experiment_b_confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

## 11. 高频误判与高置信度错误

In [ ]:
error_df = test_df[~test_df["correct"]].copy()

error_pairs = (
    error_df
    .groupby(["true_label", "pred_label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

high_confidence_errors = (
    error_df
    .sort_values("confidence", ascending=False)
    [["text", "true_label", "pred_label", "confidence"]]
    .head(40)
)

print("最常见误判方向：")
display(error_pairs.head(20))

print("置信度最高的错误：")
display(high_confidence_errors)

## 12. 自动摘要

In [ ]:
best_row = per_class_df.iloc[0]
worst_row = per_class_df.iloc[-1]
gap = best_row["class_accuracy_recall"] - worst_row["class_accuracy_recall"]

print(
    f"总体 Accuracy：{overall_df.loc[0, 'accuracy']:.2%}\n"
    f"Balanced Accuracy：{overall_df.loc[0, 'balanced_accuracy']:.2%}\n"
    f"Macro-F1：{overall_df.loc[0, 'macro_f1']:.2%}\n\n"
    f"表现最好类别：{best_row['emotion']}，Recall={best_row['class_accuracy_recall']:.2%}\n"
    f"表现最弱类别：{worst_row['emotion']}，Recall={worst_row['class_accuracy_recall']:.2%}\n"
    f"最好与最弱类别差距：{gap * 100:.2f} 个百分点\n\n"
    f"Neutral Recall：{neutral_recall:.2%}\n"
    f"Neutral Precision：{neutral_precision:.2%}\n"
    f"False Neutral Rate：{false_neutral_rate:.2%}\n"
    f"False Emotional Rate：{false_emotional_rate:.2%}\n"
    f"类别正确率卡方检验 p={p_value:.6g}"
)

## 13. 导出结果

In [ ]:
overall_df.to_csv("experiment_b_overall_metrics.csv", index=False)
per_class_df.to_csv("experiment_b_per_class_metrics.csv", index=False)
neutral_metrics_df.to_csv("experiment_b_neutral_metrics.csv", index=False)
test_df.to_csv("experiment_b_all_predictions.csv", index=False)
error_pairs.to_csv("experiment_b_error_pairs.csv", index=False)
high_confidence_errors.to_csv(
    "experiment_b_high_confidence_errors.csv", index=False
)

print("实验 B 的 CSV 与 PNG 文件已保存到当前 Colab 运行目录。")

## 报告文字模板

> 本实验使用 `j-hartmann/emotion-english-distilroberta-base` 对 MELD 官方测试集中的七类情绪进行文本分类。模型总体 Accuracy 为 `{结果}`，Balanced Accuracy 为 `{结果}`，Macro-F1 为 `{结果}`。以各类 Recall 作为类别分类准确率时，表现最佳的情绪为 `{结果}`，表现最弱的情绪为 `{结果}`，相差 `{结果}` 个百分点。Neutral 的 Precision、Recall 和 F1 分别为 `{结果}`、`{结果}` 和 `{结果}`。False Neutral Rate 为 `{结果}`，表明真实非中性语料中有该比例被漏判为中性；False Emotional Rate 为 `{结果}`，表明真实中性语料中有该比例被过度解读为某种情绪。由于该模型训练数据包含 MELD，本结果主要用于分析类别间性能差异和错误结构。